<a href="https://colab.research.google.com/github/2022311057/Graduation_research/blob/main/Part_of_speech_structure_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import pandas as pd

# ---------------------------------------------------------
# 1. データ定義 (Data Setup)
# ---------------------------------------------------------

# 品詞リスト
POS_LIST = [
    "名詞", "動詞", "形容詞", "助詞", "助動詞",
    "副詞", "接続詞", "連体詞", "感動詞", "Others"
]

# 文頭確率 (Start Probabilities)
# ※以前の画像データに基づく値を使用
START_PROBS = {
    "名詞": 0.701754386,
    "動詞": 0.0350877193,
    "形容詞": 0.05263157895,
    "助詞": 0.1052631579,
    "助動詞": 0.0,
    "副詞": 0.01754385965,
    "接続詞": 0.08771929825,
    "連体詞": 0.0,
    "感動詞": 0.0,
    "Others": 0.0
}

# 文末確率 (End Probabilities)
# ※以前の画像データに基づく値を使用
END_PROBS = {
    "名詞": 0.1785714286,
    "動詞": 0.2321428571,
    "形容詞": 0.0,
    "助詞": 0.3392857143,
    "助動詞": 0.2142857143,
    "副詞": 0.0,
    "接続詞": 0.03571428571,
    "連体詞": 0.0,
    "感動詞": 0.0,
    "Others": 0.0
}

# 遷移行列 (Transition Matrix)
# アップロードされた transition_matrix.csv の値を反映
TRANS_MATRIX = {
    "名詞": {
        "名詞": 0.1259259259259259,
        "動詞": 0.0444444444444444,
        "形容詞": 0.0,
        "助詞": 0.725925925925926,
        "助動詞": 0.0962962962962963,
        "副詞": 0.0,
        "接続詞": 0.0,
        "連体詞": 0.0,
        "感動詞": 0.0,
        "Others": 0.0074074074074074
    },
    "動詞": {
        "名詞": 0.1578947368421052,
        "動詞": 0.1929824561403508,
        "形容詞": 0.0,
        "助詞": 0.4035087719298245,
        "助動詞": 0.2456140350877192,
        "副詞": 0.0,
        "接続詞": 0.0,
        "連体詞": 0.0,
        "感動詞": 0.0,
        "Others": 0.0
    },
    "形容詞": {
        "名詞": 0.8333333333333334,
        "動詞": 0.0,
        "形容詞": 0.0,
        "助詞": 0.1666666666666666,
        "助動詞": 0.0,
        "副詞": 0.0,
        "接続詞": 0.0,
        "連体詞": 0.0,
        "感動詞": 0.0,
        "Others": 0.0
    },
    "助詞": {
        "名詞": 0.4090909090909091,
        "動詞": 0.3636363636363636,
        "形容詞": 0.0227272727272727,
        "助詞": 0.1287878787878787,
        "助動詞": 0.0378787878787878,
        "副詞": 0.0378787878787878,
        "接続詞": 0.0,
        "連体詞": 0.0,
        "感動詞": 0.0,
        "Others": 0.0
    },
    "助動詞": {
        "名詞": 0.6,
        "動詞": 0.1,
        "形容詞": 0.0,
        "助詞": 0.3,
        "助動詞": 0.0,
        "副詞": 0.0,
        "接続詞": 0.0,
        "連体詞": 0.0,
        "感動詞": 0.0,
        "Others": 0.0
    },
    "副詞": {
        "名詞": 0.6666666666666666,
        "動詞": 0.1666666666666666,
        "形容詞": 0.0,
        "助詞": 0.1666666666666666,
        "助動詞": 0.0,
        "副詞": 0.0,
        "接続詞": 0.0,
        "連体詞": 0.0,
        "感動詞": 0.0,
        "Others": 0.0
    },
    "接続詞": {
        "名詞": 1.0,
        "動詞": 0.0,
        "形容詞": 0.0,
        "助詞": 0.0,
        "助動詞": 0.0,
        "副詞": 0.0,
        "接続詞": 0.0,
        "連体詞": 0.0,
        "感動詞": 0.0,
        "Others": 0.0
    },
    "連体詞": {
        "名詞": 0.0,
        "動詞": 0.0,
        "形容詞": 0.0,
        "助詞": 0.0,
        "助動詞": 0.0,
        "副詞": 0.0,
        "接続詞": 0.0,
        "連体詞": 0.0,
        "感動詞": 0.0,
        "Others": 0.0
    },
    "感動詞": {
        "名詞": 0.0,
        "動詞": 0.0,
        "形容詞": 0.0,
        "助詞": 0.0,
        "助動詞": 0.0,
        "副詞": 0.0,
        "接続詞": 0.0,
        "連体詞": 0.0,
        "感動詞": 0.0,
        "Others": 0.0
    },
    "Others": {
        "名詞": 1.0,
        "動詞": 0.0,
        "形容詞": 0.0,
        "助詞": 0.0,
        "助動詞": 0.0,
        "副詞": 0.0,
        "接続詞": 0.0,
        "連体詞": 0.0,
        "感動詞": 0.0,
        "Others": 0.0
    }
}

# ---------------------------------------------------------
# 2. 関数定義 (Logic)
# ---------------------------------------------------------

def select_next_pos(prev_pos, is_last_col):
    """
    前の品詞(prev_pos)を受け取り、次の品詞を確率的に決定して返す。
    is_last_col=True の場合は、文末確率も考慮する。
    """
    candidates = POS_LIST
    weights = []

    # A. 1列目（文頭）の場合：文頭確率を使用
    if prev_pos is None:
        weights = [START_PROBS.get(pos, 0) for pos in candidates]

    # B. 2~3列目（途中）の場合：遷移確率を使用
    elif not is_last_col:
        trans_row = TRANS_MATRIX.get(prev_pos, {p: 0 for p in POS_LIST})
        weights = [trans_row.get(pos, 0) for pos in candidates]

    # C. 4列目（文末）の場合：遷移確率 × 文末確率 の積を使用
    else:
        trans_row = TRANS_MATRIX.get(prev_pos, {p: 0 for p in POS_LIST})
        for pos in candidates:
            p_trans = trans_row.get(pos, 0)
            p_end = END_PROBS.get(pos, 0)

            # 【ここがポイント】確率の積をスコアとする
            score = p_trans * p_end
            weights.append(score)

    # 確率の合計が0になってしまった場合の回避
    if sum(weights) == 0:
        # 連体詞や感動詞のように遷移先がすべて0の場合の救済措置
        return "Error"

    # ルーレット選択 (weightsに基づいてランダムに1つ選ぶ)
    selected = random.choices(candidates, weights=weights, k=1)[0]
    return selected

def generate_pos_grid():
    """4行×4列の品詞グリッドを生成する"""
    grid = []

    for row_idx in range(4): # 4行作成
        current_row = []
        prev_pos = None # 行が変わるたびにリセット

        for col_idx in range(4): # 4列作成
            # 最後の列（indexが3）かどうかを判定
            is_last = (col_idx == 3)

            # 次の品詞を決定
            new_pos = select_next_pos(prev_pos, is_last_col=is_last)

            current_row.append(new_pos)
            prev_pos = new_pos # 次のステップのために現在の品詞を保存

        grid.append(current_row)

    return grid

# ---------------------------------------------------------
# 3. 実行と出力 (Execution)
# ---------------------------------------------------------

# グリッド生成
lyric_structure = generate_pos_grid()

# Pandasで見やすく表示
df_output = pd.DataFrame(lyric_structure, columns=["Beat 1", "Beat 2", "Beat 3", "Beat 4"])
print("=== 生成された歌詞構造（品詞グリッド） ===")
print(df_output)

=== 生成された歌詞構造（品詞グリッド） ===
  Beat 1 Beat 2 Beat 3 Beat 4
0     名詞     助詞    助動詞     助詞
1     名詞     助詞     名詞     助詞
2    形容詞     名詞     助詞     動詞
3    形容詞     名詞     助詞     助詞
